In [1]:
from freqtrade.configuration import Configuration
from pathlib import Path
import os
from freqtrade.data.history import load_pair_history
from freqtrade.enums import CandleType
from freqtrade.resolvers import StrategyResolver
from freqtrade.data.dataprovider import DataProvider
from plotly import graph_objects as go
from freqtrade.plot.plotting import  generate_candlestick_graph
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
import numpy as np

In [2]:
project_root = "somedir/freqtrade"
i=0
try:
    os.chdirdir(project_root)
    assert Path('docker-compose.yml').is_file()
except:
    while i<4 and (not Path('docker-compose.yml').is_file()):
        os.chdir(Path(Path.cwd(), '../'))
        i+=1
    project_root = Path.cwd()
print(Path.cwd())

/root/trade


In [3]:
config = Configuration.from_files(["user_data/kucoin_config.json"])
config["strategy"] = "HoldStrategy"
data_location = config["datadir"]
# strategy = StrategyResolver.load_strategy(config)
# strategy.dp = DataProvider(config, None, None)
# strategy.ft_bot_start()

In [4]:
start_time = '20240118-'
pairs = config.get('exchange').get('pair_whitelist')
timeframe = config.get('timeframe')
pair = pairs[0]
pair

'WIF/USDT'

In [ ]:
!freqtrade download-data -c user_data/kucoin_config.json --timeframe $timeframe

In [5]:
dataframe = load_pair_history(
        datadir=data_location,
        timeframe=timeframe,
        pair=pair,
        data_format = "feather",
        candle_type=CandleType.SPOT,
    )
# df = strategy.analyze_ticker(dataframe, {'pair': pair})

In [6]:
def cluster(dataframe):
    X = dataframe['close'].values.reshape(-1,1)
    kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
    return kmeans.predict(X)

In [7]:
dataframe['label_1'] = cluster(dataframe)

grouped = dataframe.groupby(['label_1']).apply(cluster).to_dict()
for c, values in grouped.items():
    condition_1 = dataframe['label_1'] == c
    dataframe.loc[condition_1, 'label_2'] = values

grouped = dataframe.groupby(['label_1','label_2']).apply(cluster).to_dict()
for c, values in grouped.items():
    condition_1 = dataframe['label_1'] == c[0]
    condition_2 = dataframe['label_2'] == c[1]
    dataframe.loc[(condition_1 & condition_2), 'label_3'] = values

dataframe


,date,open,high,low,close,volume,label_1,label_2,label_3
0,2024-01-18 13:00:00+00:00,0.4780,0.4800,0.4100,0.4236,575523.7451,0,2.0,1.0
1,2024-01-18 13:15:00+00:00,0.4230,0.4300,0.4111,0.4172,58270.6912,0,2.0,2.0
2,2024-01-18 13:30:00+00:00,0.4141,0.6900,0.4136,0.4390,762385.7196,0,2.0,1.0
3,2024-01-18 13:45:00+00:00,0.4390,0.4485,0.4360,0.4392,269356.9719,0,2.0,1.0
4,2024-01-18 14:00:00+00:00,0.4407,0.4540,0.4367,0.4399,199967.5531,0,2.0,1.0
...,...,...,...,...,...,...,...,...,...
6400,2024-03-25 05:00:00+00:00,2.7770,2.7770,2.7418,2.7483,45511.6483,1,2.0,1.0
6401,2024-03-25 05:15:00+00:00,2.7476,2.7732,2.7322,2.7614,29323.9035,1,2.0,1.0
6402,2024-03-25 05:30:00+00:00,2.7583,2.7943,2.7560,2.7782,70133.5634,1,2.0,1.0
6403,2024-03-25 05:45:00+00:00,2.7775,2.8100,2.7744,2.7899,29211.4658,1,2.0,1.0


In [9]:
mins = dataframe.groupby([f'label_{i}' for i in [1,2,3]]).min().close.sort_values().values
mins

array([0.1725, 0.2128, 0.258 , 0.3043, 0.356 , 0.4229, 0.5606, 0.7044,
       0.8358, 0.9738, 1.1205, 1.2569, 1.409 , 1.519 , 1.6119, 1.7058,
       1.7921, 1.8909, 2.0149, 2.1626, 2.2914, 2.4477, 2.5928, 2.7382,
       2.9028, 3.0862, 3.2427])

In [10]:
mins[mins < 2.0149][-1]

1.8909

In [11]:
mins[mins > 2.0149][0]

2.1626

In [ ]:
# df = df[df.date<='20240324 19:30']
labels = [1,2,3]
colors = ['green', 'red', 'orange']
fig = generate_candlestick_graph(pair=pair, data=dataframe)

max_values = dataframe[[f'label_1', 'close']].groupby(['label_1']).max().close.values
min_values = dataframe[[f'label_1', 'close']].groupby(['label_1']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=2, line_color='green')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=2, line_color='green')

max_values = dataframe[[f'label_1', 'label_2', 'close']].groupby(['label_1', 'label_2']).max().close.values
min_values = dataframe[[f'label_1', 'label_2', 'close']].groupby(['label_1', 'label_2']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=1, line_color='red')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=1, line_color='red')

max_values = dataframe[[f'label_1', 'label_2', 'label_3', 'close']].groupby(['label_1', 'label_2', 'label_3']).max().close.values
min_values = dataframe[[f'label_1', 'label_2', 'label_3', 'close']].groupby(['label_1', 'label_2', 'label_3']).min().close.values
for max_value in max_values:
    fig.add_hline(y=max_value, line_width=0.5, line_color='orange')
for min_value in min_values:
    fig.add_hline(y=min_value, line_width=0.5, line_color='orange')

fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [70]:
dataframe = load_pair_history(
        datadir=data_location,
        timeframe=timeframe,
        pair=pair,
        data_format = "feather",
        candle_type=CandleType.SPOT,
    )
df = strategy.analyze_ticker(dataframe, {'pair': pair})

In [71]:
labels = [1,2,3]
colors = ['green', 'red', 'orange']
fig = generate_candlestick_graph(pair=pair, data=df)
for label, color in zip(labels, colors):
    max_values = df[[f'label_{label}', 'close']].groupby([f'label_{label}']).max().close.values
    min_values = df[[f'label_{label}', 'close']].groupby([f'label_{label}']).min().close.values
    for max_value in max_values:
        fig.add_hline(y=max_value, line_width=1/label, line_color=color)
    for min_value in min_values:
        fig.add_hline(y=min_value, line_width=1/label, line_color=color)
fig.update_layout(autosize=True, height=800, xaxis=dict(rangeslider=dict(visible=False)))

In [ ]:
def populate_indicators(self, dataframe: DataFrame, metadata: dict) -> DataFrame:
    dataframe_ = dataframe.copy()
    for c in range(1,4):
        X = dataframe_['close'].values.reshape(-1,1)
        kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
        dataframe_[f'label_{c}'] = kmeans.predict(X)
        dataframe[f'label_{c}'] = dataframe_[f'label_{c}'].astype(str)
        labels_sorted = dataframe[[f'label_{c}','close']].groupby(f'label_{c}').min()
        labels_sorted = labels_sorted.sort_values(by=['close']).reset_index().to_dict('index')
        labels_sorted = {value[f'label_{c}']:f'cluster_{key}' for key, value in labels_sorted.items()}
        dataframe = dataframe.replace({f'label_{c}':labels_sorted})
        X = dataframe_.index.values.reshape(-1,1)
        y = dataframe_.close.values
        model = LinearRegression()
        model.fit(X, y)
        dataframe[f'label_{c}_coef'] = model.coef_[0]
        dataframe_ = dataframe_[dataframe_[f'label_{c}']==dataframe_[f'label_{c}'].iat[-1]]

    return dataframe